In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 712e1f1c-65be-4ba4-8fb2-3660b42b59e1, 3, Finished, Available, Finished, False)

In [2]:
# Create custom schemas in Lakehouse
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, 712e1f1c-65be-4ba4-8fb2-3660b42b59e1, 4, Finished, Available, Finished, False)

DataFrame[]

In [3]:
# Bronze Layer — Ingest raw CSVs into Delta tables
# Use multiLine + quote + escape for ALL files to handle embedded newlines/quotes safely
# This preserves all rows without corruption

temp = "Files/raw/"

def read_raw_csv(filename):
    return (
        spark.read
        .option("header", "true")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .option("inferSchema", "true")
        .csv(f"{temp}{filename}")
    )

jobs        = read_raw_csv("jobs.csv")
locations   = read_raw_csv("job_locations.csv")
occupations = read_raw_csv("occupations.csv")
skills      = read_raw_csv("requirement_skills.csv")
channels    = read_raw_csv("channels.csv")
messages    = read_raw_csv("messages.csv")
keywords    = read_raw_csv("search_keywords.csv")

# Write to Bronze schema
jobs.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.jobs")
locations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.locations")
occupations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.occupations")
skills.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.skills")
channels.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.channels")
messages.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.messages")
keywords.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.keywords")

print("Successfully ingested into Bronze schema:")
for table_name in ["jobs", "locations", "occupations", "skills", "channels", "messages", "keywords"]:
    cnt = spark.read.table(f"bronze.{table_name}").count()
    print(f"  bronze.{table_name}: {cnt} rows")

spark.read.table("bronze.jobs").printSchema()
print("--- Bronze Layer Completed ---")

StatementMeta(, 712e1f1c-65be-4ba4-8fb2-3660b42b59e1, 5, Finished, Available, Finished, False)

Successfully ingested into Bronze schema:
  bronze.jobs: 22849 rows
  bronze.locations: 22846 rows
  bronze.occupations: 30896 rows
  bronze.skills: 50288 rows
  bronze.channels: 21 rows
  bronze.messages: 21516 rows
  bronze.keywords: 89728 rows
root
 |-- id: integer (nullable = true)
 |-- is_job_vacancy: integer (nullable = true)
 |-- message_id: integer (nullable = true)
 |-- job_name: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- job_salary: string (nullable = true)
 |-- job_descriptions: string (nullable = true)
 |-- input_language: string (nullable = true)
 |-- job_date: timestamp (nullable = true)
 |-- is_active: integer (nullable = true)
 |-- is_verified: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)

--- Bronze Layer Completed ---
